# Fases 1 a 5b — Datos sintéticos, diálogos y control de calidad
**POC Speech Analytics · Retención de desafiliaciones · Seguro vehicular (caso ficticio)**

Este notebook corre en Google Colab. Orden:
1. Cargar el proyecto
2. Generar las tablas operativas y la verdad de terreno
3. Configurar la clave de OpenAI (con *Secrets* de Colab, nunca pegada en el código)
4. Probar con 5 diálogos y revisarlos a mano
5. Generar la muestra completa (402 diálogos)
5b. Normalizar y controlar la calidad de los diálogos


### **1. Cargar el proyecto**
Clona el repositorio. **No lo ejecutes dos veces** en la misma sesión: crearía una carpeta anidada. Si la carpeta ya existe, usa `%cd /content/speech-analytics-retencion-autos`.

In [ ]:
!git clone https://github.com/made15pc/speech-analytics-retencion-autos.git
%cd speech-analytics-retencion-autos
!pip install -q -r requirements.txt

### **Recuperación (solo si algo falla)**
Si el kernel se reinicia y un `import` dice `No module named 'generar_dialogos'`, ejecuta esta celda: fija la ruta de forma absoluta.

In [ ]:
import os, sys
os.chdir("/content/speech-analytics-retencion-autos")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd(), "| script existe:", os.path.exists("src/generar_dialogos.py"))

### **2. Generar datos sintéticos y revisar KPIs**
Toma ~5 segundos. La semilla fija hace que siempre salgan los mismos datos.

In [ ]:
!python src/generar_datos.py --salida data/raw --semilla 42
!python src/revisar_kpis.py --datos data/raw

### **3. Clave de OpenAI**
En Colab: ícono de llave 🔑 (Secrets) → *Add new secret* → nombre `OPENAI_API_KEY` → pega tu clave → activa *Notebook access*.

**Ojo:** nunca escribas la clave en una celda; si subes el notebook a GitHub quedaría expuesta.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from openai import OpenAI
modelos = sorted(m.id for m in OpenAI().models.list().data)
print([m for m in modelos if any(k in m for k in ("luna", "mini", "transcribe", "tts", "whisper"))])

Elige el modelo de texto más barato disponible en tu cuenta (por defecto el script usa `gpt-6-luna`). Si no aparece, cambia la variable `MODELO`.

In [ ]:
MODELO = "gpt-6-luna"   # cámbialo si tu cuenta no lo tiene

### **4. Prueba piloto con 5 diálogos**
Antes de gastar en 400, revisa que el diálogo **cumpla la ficha**: si la ficha dice que el asesor no indaga, no debe indagar. Esta revisión manual es parte del método.

In [ ]:
!python src/generar_dialogos.py --n 5 --modelo $MODELO --salida data/dialogos_piloto

In [ ]:
import os, sys, json, importlib
import pandas as pd
os.chdir("/content/speech-analytics-retencion-autos")
sys.path.insert(0, os.path.abspath("src"))

import generar_dialogos
importlib.reload(generar_dialogos)   # fuerza a leer el archivo actual
from generar_dialogos import construir_ficha

muestra = pd.read_csv("data/dialogos_piloto/muestra.csv").set_index("llamada_id")
dialogos = [json.loads(l) for l in open("data/dialogos_piloto/dialogos.jsonl", encoding="utf-8")]

d = dialogos[0]
print(construir_ficha(muestra.loc[d["llamada_id"]].rename(d["llamada_id"])))
print("\n" + "=" * 80 + "\n")
for t in d["turnos"]:
    print(f"{t['hablante'].upper():8} {t['texto']}")

### **5. Generación completa (~400 llamadas)**
Si se corta, vuelve a ejecutar: el script retoma donde quedó. Al final imprime los tokens usados para que calcules el costo real.

In [ ]:
!python src/generar_dialogos.py --n 400 --modelo $MODELO --salida data/dialogos --hilos 6

### **5b. Normalizar los diálogos generados**
Corrige sin volver a llamar a la API tres defectos observados en la corrida real: turnos con la clave `text`, marcadores inventados para números de gestión (`[NÚMERO DE SOLICITUD]`) y acotaciones escénicas (`[Pausa breve]`). Deja una copia `.antes_de_normalizar` y es idempotente.

In [ ]:
!python src/normalizar_dialogos.py --dialogos data/dialogos/dialogos.jsonl

In [ ]:
dialogos = [json.loads(l) for l in open("data/dialogos/dialogos.jsonl", encoding="utf-8")]
ok = [d for d in dialogos if d.get("turnos")]
palabras = pd.Series([sum(len(t["texto"].split()) for t in d["turnos"]) for d in ok])
print(f"Diálogos válidos: {len(ok)} de {len(dialogos)}")
print(palabras.describe().round(0))

### **5b. Control automático de calidad**
Separa **fallos duros** (etiquetas técnicas filtradas, marcadores inválidos, fechas fuera del escenario, estructura rota) de **alertas blandas** (heurísticas que sirven para elegir qué leer, no para descartar). Resultado de la corrida de referencia: 1 fallo duro aceptado (L004591), 22 diálogos con alerta útil, 380 sin alerta.

In [ ]:
!python src/control_calidad.py --dialogos data/dialogos/dialogos.jsonl --muestra data/dialogos/muestra.csv --salida data/dialogos/qa

### **Resultado final del QA**
`menciona_ficha` (lenguaje normal de call center) y `no_debia_aceptar_pero_parece_aceptar` (precisión muy baja: las negaciones y modismos engañan a una regex) se descartan. El resto queda como columna `con_alerta` para un análisis de sensibilidad en la validación.

In [ ]:
import pandas as pd
qa = pd.read_csv("data/dialogos/qa/qa_detalle.csv").fillna("")
ignorar = {"no_debia_aceptar_pero_parece_aceptar", "menciona_ficha"}
qa["alertas_utiles"] = qa.blandos.apply(lambda s: ";".join(x for x in s.split(";") if x and x not in ignorar))
qa["con_alerta"] = qa.alertas_utiles != ""
qa[["llamada_id", "duros", "blandos", "alertas_utiles", "con_alerta"]].to_csv("data/dialogos/qa/qa_final.csv", index=False)
print(qa.con_alerta.value_counts().to_dict(), "| total:", len(qa))
print("Fallo duro restante:", qa.loc[qa.duros != "", ["llamada_id", "duros"]].values.tolist())

### **6. Guardar en Google Drive**
Colab borra los archivos al cerrar la sesión.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/speech-analytics-retencion-autos
!cp -r data /content/drive/MyDrive/speech-analytics-retencion-autos/